## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-00-54-27 +0000,nyt,Watch: Christine Lagarde Wants to Fight Fake News,https://www.nytimes.com/2026/11/18/business/de...
1,2026-11-19-00-52-56 +0000,nyt,"Trump, Xi and Putin Shake Hands as Leaders Gat...",https://www.nytimes.com/live/2026/11/18/world/...
2,2026-11-19-00-52-54 +0000,missionlocal,"Case of Troy McAlister, S.F. man whose deadly ...",https://missionlocal.org/2026/11/case-of-troy-...
3,2026-11-19-00-52-53 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/18/world/...
4,2026-11-19-00-52-52 +0000,nyt,The Shadow of Yalta Hangs Over Trump’s Summit ...,https://www.nytimes.com/2026/11/18/world/asia/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2556/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
6,trump,62
97,nvidia,24
52,new,23
237,tech,20
86,up,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
219,2026-11-18-18-41-56 +0000,chitri,President Trump plans separate talks with Vlad...,https://www.chicagotribune.com/2026/11/18/trum...,153
136,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,152
328,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...,139
274,2026-11-18-16-10-44 +0000,bbc,Trump shakes hands with Putin and Xi ahead of ...,https://www.bbc.co.uk/news/articles/ck9d75x5ww...,135
217,2026-11-18-18-52-04 +0000,nyt,Trump Arrives in China Focused on Friendship W...,https://www.nytimes.com/2026/11/18/us/politics...,134


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
219,153,2026-11-18-18-41-56 +0000,chitri,President Trump plans separate talks with Vlad...,https://www.chicagotribune.com/2026/11/18/trum...
421,110,2026-11-18-04-23-00 +0000,wsj,Democrat Mary Peltola unseated Republican Sen....,https://www.wsj.com/politics/elections/democra...
319,79,2026-11-18-12-56-00 +0000,wsj,The Morning Risk Report: SEC Is Probing Nvidia...,https://www.wsj.com/risk-compliance-journal/th...
136,77,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
18,63,2026-11-19-00-30-02 +0000,startribune,"Anthropic prices its IPO at $83 per share, val...",https://www.startribune.com/anthropic-prices-i...
41,57,2026-11-18-23-47-13 +0000,nypost,Veteran NYPD school safety agent strangled by ...,https://nypost.com/2026/11/18/us-news/veteran-...
422,53,2026-11-18-04-00-54 +0000,bbc,Jeff Bezos says he was 'heavily involved' in c...,https://www.bbc.co.uk/news/articles/cm5y1eejk2...
61,50,2026-11-18-23-11-17 +0000,nypost,Iran and US trade tit-for-tat strikes as wild ...,https://nypost.com/2026/11/18/world-news/uae-o...
125,50,2026-11-18-21-53-00 +0000,wsj,"Tech, Media & Telecom Roundup: Market Talk",https://www.wsj.com/business/tech-media-teleco...
366,45,2026-11-18-10-23-57 +0000,bbc,Democrats win Alaska Senate race two weeks aft...,https://www.bbc.co.uk/news/articles/cqev3v2ygz...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
